In [1]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Model Selection and Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Model Serialization
import joblib

# Set visual style for plots
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
print("Libraries successfully imported!")

Libraries successfully imported!


## 2. Load the Dataset


In [2]:
# Load the dataset
df = pd.read_csv('../data/airline_passenger_satisfaction.csv')

# Display shape and first few rows
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (15000, 23)


,Gender,Customer_Type,Age,Type_of_Travel,Class,Flight_Distance,Inflight_wifi_service,Departure_Arrival_time_convenient,Ease_of_Online_booking,Gate_location,...,Inflight_entertainment,On-board_service,Leg_room_service,Baggage_handling,Checkin_service,Inflight_service,Cleanliness,Departure_Delay_in_Minutes,Arrival_Delay_in_Minutes,satisfaction
0,Male,Loyal Customer,19,Business travel,Business,2601,3,3,4,4,...,5,5,3,4,5,5,5,0,1,satisfied
1,Male,Loyal Customer,44,Business travel,Eco,1821,2,2,1,3,...,2,2,2,3,2,4,3,0,0,neutral_or_dissatisfied
2,Male,disloyal Customer,47,Business travel,Eco,1311,4,5,2,3,...,4,3,4,3,5,2,2,62,61,neutral_or_dissatisfied
3,Male,Loyal Customer,20,Business travel,Business,1054,2,2,4,2,...,3,3,5,2,1,2,0,0,8,neutral_or_dissatisfied
4,Male,Loyal Customer,43,Business travel,Eco,1537,4,4,5,3,...,4,0,4,5,4,5,4,0,0,satisfied


## 3. Exploratory Data Analysis (EDA) 


In [3]:
df.shape()

TypeError: 'tuple' object is not callable

### Plot 1: Target Variable Class Distribution


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='satisfaction', data=df, hue='satisfaction', palette='viridis', legend=False)
plt.title('Passenger Satisfaction Distribution (Target)')
plt.xlabel('Satisfaction Status')
plt.ylabel('Passenger Count')
plt.show()

print(df['satisfaction'].value_counts())
print("\nClass Distribution Percentage:")
print(df['satisfaction'].value_counts(normalize=True) * 100)

**Insight 1:** The dataset is semi-balanced with **57.2% Neutral/Dissatisfied** and **42.8% Satisfied** passengers. Since there is no extreme class imbalance (e.g. 99% vs 1%), we can safely use **Accuracy** as a reliable evaluation metric.

### Plot 2: Satisfaction Split by Travel Class


In [ ]:
sns.countplot(data=df, x='Class', hue='satisfaction', palette=['#f87171', '#34d399'])
plt.title('Satisfaction Distribution by Ticket Class')
plt.xlabel('Travel Class')
plt.ylabel('Count')
plt.legend(title='Satisfaction')
plt.show()

**Insight 2:** Passengers in **Business Class** show an exceptionally high satisfaction rate, whereas **Economy (Eco)** and **Economy Plus** flyers are overwhelmingly dissatisfied. This highlights that cabin space, comfort, and premium treatments are strong satisfaction drivers.

### Plot 3: Satisfaction Split by Type of Travel


In [ ]:
sns.countplot(data=df, x='Type_of_Travel', hue='satisfaction', palette=['#f87171', '#34d399'])
plt.title('Satisfaction Distribution by Travel Purpose')
plt.xlabel('Type of Travel')
plt.ylabel('Count')
plt.legend(title='Satisfaction')
plt.show()

**Insight 3:** **Business travelers** are mostly satisfied, whereas **Personal/Leisure travelers** are highly dissatisfied. Personal travelers usually pay out of pocket, making them highly price-sensitive and less tolerant of lack of amenities.

### Plot 4: Passenger Age Distribution by Satisfaction


In [ ]:
sns.kdeplot(data=df, x='Age', hue='satisfaction', fill=True, common_norm=False, palette=['#f87171', '#34d399'], alpha=0.4)
plt.title('Age Density Curves for Satisfied vs Dissatisfied Passengers')
plt.xlabel('Age (Years)')
plt.ylabel('Density')
plt.show()

**Insight 4:** Satisfaction density peaks in the **35-50 age bracket**, which corresponds to frequent business flyers. Younger flyers (under 30) and elderly passengers (above 60) show higher relative rates of dissatisfaction, likely due to different comfort expectations.

### Plot 5: Outlier Detection on Continuous Numerical Columns


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(data=df, y='Age', color='skyblue', ax=axes[0])
axes[0].set_title('Age Boxplot')

sns.boxplot(data=df, y='Flight_Distance', color='lightgreen', ax=axes[1])
axes[1].set_title('Flight Distance Boxplot')

sns.boxplot(data=df, y='Arrival_Delay_in_Minutes', color='salmon', ax=axes[2])
axes[2].set_title('Arrival Delay Boxplot')

plt.tight_layout()
plt.show()

**Insight 5:** Age and Flight Distance have no extreme outliers. However, the **Arrival Delay** column is heavily populated with outliers (data points beyond the upper whisker). This is standard for flight delay data and will be handled by robust tree-based models like Random Forests which are immune to outliers.

### Plot 6: Feature Correlation Heatmap


In [ ]:
# Map columns to numbers temporarily for correlation heatmap
corr_df = df.copy()
corr_df['Gender'] = corr_df['Gender'].map({'Male': 0, 'Female': 1})
corr_df['Customer_Type'] = corr_df['Customer_Type'].map({'Loyal Customer': 0, 'disloyal Customer': 1})
corr_df['Type_of_Travel'] = corr_df['Type_of_Travel'].map({'Business travel': 0, 'Personal Travel': 1})
corr_df['Class'] = corr_df['Class'].map({'Business': 0, 'Eco': 1, 'Eco Plus': 2})
corr_df['satisfaction'] = corr_df['satisfaction'].map({'neutral_or_dissatisfied': 0, 'satisfied': 1})

# Compute correlation matrix
corr_matrix = corr_df.corr()

# Plot Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Full Feature Correlation Matrix')
plt.show()

**Insight 6:**
1. **Satisfaction correlations:** Online Boarding, Inflight Wifi, and Seat Comfort show the strongest positive correlation with passenger satisfaction.
2. **Multicollinearity:** Departure Delay and Arrival Delay show an almost perfect linear correlation (~0.98), suggesting that they contain identical information. We can keep only one or use Random Forest which natively handles collinearity.

## 4. Feature Selection using Random Forest


In [ ]:
# 1. Separate features and target
X_all = corr_df.drop('satisfaction', axis=1)
y_all = corr_df['satisfaction']

# 2. Train a temporary Random Forest model
model_temp = RandomForestClassifier(n_estimators=100, random_state=42)
model_temp.fit(X_all, y_all)

# 3. Print the importance score of each feature directly
importances = model_temp.feature_importances_
print("Feature Importances:")
for i in range(len(X_all.columns)):
    column_name = X_all.columns[i]
    score = importances[i]
    print(column_name, ":", round(score, 4))

## 5. Final Model Preprocessing & Training 

In [ ]:
selected_features = [
    'Online_boarding',
    'Cleanliness',
    'Seat_comfort',
    'Inflight_entertainment',
    'Inflight_wifi_service',
    'Ease_of_Online_booking',
    'Flight_Distance',
    'On-board_service',
    'Class',
    'Arrival_Delay_in_Minutes'
]

# Map categories to numbers for final modeling
df_final = df.copy()
df_final['Class'] = df_final['Class'].map({'Business': 0, 'Eco': 1, 'Eco Plus': 2})
df_final['satisfaction'] = df_final['satisfaction'].map({'neutral_or_dissatisfied': 0, 'satisfied': 1})

X = df_final[selected_features]
y = df_final['satisfaction']


# Train test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Top 10 features scaled successfully!")

In [ ]:
# Train Baseline Logistic Regression Model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)
print(f"Logistic Regression Test Accuracy: {accuracy_score(y_test, lr_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

In [ ]:
# Train Production Random Forest Classifier Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
print(f"Random Forest Test Accuracy: {accuracy_score(y_test, rf_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

In [ ]:
# Plot Confusion Matrix for Random Forest
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Random Forest - Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Create models folder if it does not exist
if not os.path.exists('../models'):
    os.makedirs('../models')

# Save the trained model and scaler
joblib.dump(rf_model, '../models/passenger_satisfaction_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

# Save mapping metadata for deployment references
metadata = {
    'mappings': {
        'Class': {'Business': 0, 'Eco': 1, 'Eco Plus': 2},
        'satisfaction': {'neutral_or_dissatisfied': 0, 'satisfied': 1}
    },
    'feature_names': selected_features
}
joblib.dump(metadata, '../models/metadata.pkl')

print("Model, Scaler, and Metadata successfully saved to '../models/' folder!")